# 05 · Train, analyze, and cross-predict with the recommended model

Notebook 03 established the model worth using: mask + context features (σ ≤ 240px)
and the prior-corrected two-part head (`models.recipe`/`models.train.
train_calibrated`). This notebook applies exactly that configuration, but without
the ablation -- edit the **config cell** below to point it at any well or any
`*_features.parquet` path, train, then:

- run the same model-analysis views `gnn_v1/mask_predict.ipynb` used (spatial feature
  maps, intrinsic + spatial correlation, gradient-based neighbor sensitivity by
  layer), updated to the current `models`/`tools` API, and
- cross-predict onto other images the way `gnn_v1/cross_predict.ipynb` did -- load
  the saved checkpoint's fitted scalers (never refit) and score it against a
  *different* image's cells, which is the only way to tell "generalizes" from
  "memorized this well".

Only `TRAIN_SOURCE`/`EVAL_SOURCES` need to change to run this on new data; nothing
else in the notebook assumes `PRIMARY_WELL` specifically.

In [ ]:
import sys, os
sys.path.append('../src')

%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tools.dataset import load_well, marker_positive, IF_BACKGROUNDS, PRIMARY_WELL, MATCHED_PAIR, RESULTS
from tools.spatial import PYRAMID_SCALES
import tools.evaluation as ev
import plotting as pl

pd.set_option('display.width', 160)
plt.rcParams['figure.dpi'] = 110

In [ ]:
import os, re
import torch
from models import recipe
from models.graph import border_mask
from models.train import calibrated_predict
from models.predictors import IFPredictor
from models.checkpoint import load_model, apply_prediction_data
from tools.morphology import indices_to_mask, plot_correlation_heatmap, plot_spatial_cross_correlation
from tools.dataset import STITCHED

# ---- what to run: edit these to point the notebook at new data, nothing else ----
TRAIN_SOURCE = PRIMARY_WELL          # a well name (looked up under data/stitched/) or a full path to a *_features.parquet
EVAL_SOURCES = [MATCHED_PAIR[1]]     # 0+ other images to cross-predict on once trained -- same name-or-path rule
MASK_CHANNEL = 'BMP4'                # f'{MASK_CHANNEL}_bin' is the model input; needs an f'{MASK_CHANNEL}+' column
TARGET_MARKERS = ['Sox17', 'T']      # y columns are f'{marker}_mean'
CUSTOM_BACKGROUNDS = IF_BACKGROUNDS  # {y_col: threshold} -- only W8_pattern1's thresholds are verified (see tools.dataset);
                                     # extend/override this dict before trusting AUROC/R2 on a different well

QUICK = False                # True -> few epochs, for a fast pass through the notebook
ANALYSIS_SPLIT = 'blocked'   # which trained model ('random' | 'blocked') the analysis/cross-predict sections use below

EPOCHS = 150 if QUICK else recipe.EPOCHS
PATIENCE = 150 if QUICK else recipe.PATIENCE
CACHE_SUFFIX = '_quick' if QUICK else ''  # keeps a provisional run from being reused as the real one

## Load & features

In [ ]:
def load_cell_table(source, mask_channel=MASK_CHANNEL):
    """`source` is a well name under data/stitched/, or a full path to a
    *_features.parquet file -- this is the hook for pointing the notebook at a
    different image. Mirrors `tools.dataset.load_well` but accepts any path."""
    path = source if str(source).endswith('.parquet') else os.path.join(STITCHED, f'{source}_features.parquet')
    out = pd.read_parquet(path)
    out[f'{mask_channel}_bin'] = out[f'{mask_channel}+'].astype(np.float32)
    return out

def _slug(source):
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', str(source)).strip('_')

df = load_cell_table(TRAIN_SOURCE)
centroids = df[['centroid_y', 'centroid_x']].to_numpy(np.float32)
IF_COLS = [f'{m}_mean' for m in TARGET_MARKERS]
TRAIN_SLUG = _slug(TRAIN_SOURCE)  # e.g. PRIMARY_WELL itself, unchanged -- matches 03/04's checkpoint naming exactly

X_MASK = recipe.mask_features(df, f'{MASK_CHANNEL}_bin', max_sigma=recipe.MAX_SIGMA, geometry=True)
print(f'{len(df):,} cells from {TRAIN_SOURCE}')
print(f'feature set ({len(X_MASK)}):')
for c in X_MASK: print('   ', c)

In [ ]:
shape = (int(centroids[:,0].max())+2, int(centroids[:,1].max())+2)
valid = np.where(~border_mask(centroids, shape, recipe.RADIUS))[0]

SPLITS = {
    'random':  tuple(valid[i] for i in ev.random_split(len(valid), seed=0)),
    'blocked': tuple(valid[i] for i in ev.spatial_block_split(centroids[valid], block=800.0, seed=0)),
}
N = len(df)
MASKS = {k: tuple(indices_to_mask(i, N) for i in v) for k, v in SPLITS.items()}
for k, (tr, va, te) in SPLITS.items():
    print(f'{k:8s} train/val/test = {len(tr):,}/{len(va):,}/{len(te):,}')

## Train

The recommended config directly -- no ablation here, 03 already made the case for
it. Both splits are fit for every marker so the random/blocked gap stays visible;
`ANALYSIS_SPLIT` picks which of the two the sections below dig into.

Cached the same way 03/04 cache theirs (`recipe.load_or_fit`): with the default
`TRAIN_SOURCE = PRIMARY_WELL`, this loads 03's checkpoints straight off disk instead
of retraining. Pointed at a different well, there's nothing to reuse, so it trains
and caches fresh -- and a second run against that same well would then hit its own
cache.

In [ ]:
ceilings = None
try:
    ceilings = pd.read_pickle(f'{RESULTS}/oracle_ceilings.pkl')
except FileNotFoundError:
    pass  # only meaningful when TRAIN_SOURCE is the well notebook 01 was run on

rows, fitted = [], {}
for marker in TARGET_MARKERS:
    y_col = f'{marker}_mean'
    for split_name, (tr, va, te) in SPLITS.items():
        trm, vam, tem = MASKS[split_name]
        cache_path = f'{RESULTS}/models/{TRAIN_SLUG}_mask_{marker}_{split_name}{CACHE_SUFFIX}_gnn.pt'
        model, data, log_w, history = recipe.load_or_fit(
            cache_path, df, X_MASK, y_col, trm, vam, tr, IF_COLS, custom_background=CUSTOM_BACKGROUNDS,
            epochs=EPOCHS, patience=PATIENCE, verbose=False)
        pred = recipe.predict(model, data, log_w)
        y = data['y'].numpy().ravel(); ypos = data['y_positive'].numpy().ravel()
        met = ev.evaluate(y[tem.numpy()], pred[tem.numpy()], ypos[tem.numpy()])
        rows.append(dict(target=marker, split=split_name, **met))
        fitted[(marker, split_name)] = dict(model=model, data=data, log_w=log_w, history=history,
                                            pred=pred, y=y, ypos=ypos, metrics=met)
        print(f"{marker:6s} {split_name:8s} R2={met['r2']:+.4f}  "
              f"AUROC={met.get('auroc', float('nan')):.3f}  AP={met.get('ap', float('nan')):.3f}")

results = pd.DataFrame(rows)
results.round(4)

In [ ]:
for marker in TARGET_MARKERS:
    sub = results[results.target == marker]
    ceiling = ceilings[marker]['r2'] if ceilings is not None and marker in ceilings else None
    pl.plot_metric_comparison(sub, metric='r2', group='target', hue='split',
                              ceiling=ceiling, title=f'{marker}: R2, random vs blocked split')
    plt.show()

## The trained model

In [ ]:
for marker in TARGET_MARKERS:
    out = fitted[(marker, ANALYSIS_SPLIT)]
    ceiling = ceilings[marker]['r2'] if ceilings is not None and marker in ceilings else None
    if out['history'] is not None:
        pl.plot_training_curves(out['history'], ceiling=ceiling, title=f'{marker} ({ANALYSIS_SPLIT} split)')
        plt.show()
    else:
        print(f'{marker}: loaded from a cached checkpoint -- no training-curve history to plot')

    tem = MASKS[ANALYSIS_SPLIT][2].numpy()
    fig, ax = plt.subplots(figsize=(5.8, 5.8))
    pl.plot_pred_vs_actual(out['y'][tem], out['pred'][tem], out['ypos'][tem], ax=ax,
                           title=f"{marker}: R2={out['metrics']['r2']:.3f}  "
                                 f"AUROC={out['metrics'].get('auroc', float('nan')):.3f}")
    plt.tight_layout(); plt.show()

    pl.plot_prediction_panel(df, f'{MASK_CHANNEL}_bin', out['y'], out['pred'], target_name=marker,
                             mask_name=f'{MASK_CHANNEL} mask (input)', smooth_sigma=60)
    plt.show()

## Model analysis

Same questions `gnn_v1/mask_predict.ipynb` asked of the earlier model, re-run against
the current `x_cols`/API: what do the scaled inputs look like in space, how do they
correlate with each other (same cell, and spatially with a neighbor), and -- since
`WeightedRadiusConv`'s edge weight is a fixed function of distance with nothing
learned to read off directly -- a gradient-based sensitivity in place of GAT-style
attention (`IFPredictor.message_sensitivity_by_layer`).

In [ ]:
ANALYSIS_MARKER = TARGET_MARKERS[0]
out = fitted[(ANALYSIS_MARKER, ANALYSIS_SPLIT)]
model, data = out['model'], out['data']
x_cols = data['x_cols']

feat = pd.DataFrame(data['x'].numpy(), columns=x_cols)
feat[f'{ANALYSIS_MARKER}_mean (scaled)'] = data['y'].numpy().ravel()
feat_cols = list(feat.columns)

ncols = 4
nrows = int(np.ceil(len(feat_cols) / ncols))
fig, axs = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 5.5 * nrows), squeeze=False)
axs = axs.ravel()
for ax, col in zip(axs, feat_cols):
    sc = ax.scatter(df['centroid_x'], df['centroid_y'], c=feat[col], cmap='coolwarm', s=2)
    ax.set_title(col, fontsize=10)
    ax.set_aspect('equal', adjustable='box')
    ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
for ax in axs[len(feat_cols):]:
    ax.set_visible(False)
fig.suptitle(f'{ANALYSIS_MARKER}: scaled model inputs + target, in space')
fig.tight_layout()
plt.show()

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 6))
plot_correlation_heatmap(feat, feat_cols, title='Pearson correlation (intrinsic, same cell)',
                         corr_type='pearson', ax=axs[0])
plot_spatial_cross_correlation(feat, feat_cols, data['edge_index'], weights=data['edge_weight'],
                               title="Pearson spatial cross-corr. (extrinsic, neighbor)",
                               corr_type='pearson', ax=axs[1])
plt.tight_layout(); plt.show()

### Gradient-based neighbor sensitivity

For each layer, the per-edge message `edge_weight * lin(x_j)`, graded by how much
the *actual* prediction moved if that message changed (grad × message -- the
standard attribution choice, not the raw gradient alone). Correlated against edge
distance and a handful of representative features -- not every `x_col`, since
`X_MASK` can carry a dozen-plus pyramid scales and plotting all of them by layer
would be more panels than anyone reads.

In [ ]:
layer_sensitivities = model.message_sensitivity_by_layer(data['x'], data['edge_index'], data['edge_weight'])

centroids_t = torch.from_numpy(centroids.copy())
src, dst = data['edge_index']
edge_dist = torch.norm(centroids_t[src] - centroids_t[dst], dim=1)

sample_cols = list(dict.fromkeys([x_cols[0]] + x_cols[1::max(1, len(x_cols) // 3)][:3]))

n_panels = 1 + len(sample_cols)
n_layers = len(layer_sensitivities)
fig, axes = plt.subplots(n_layers, n_panels, figsize=(5 * n_panels, 5 * n_layers), sharey=True, squeeze=False)

for layer_idx, layer_sens in enumerate(layer_sensitivities):
    ax = axes[layer_idx, 0]
    ax.scatter(edge_dist.numpy(), layer_sens.numpy(), s=2, alpha=0.15, linewidths=0)
    corr_d = np.corrcoef(edge_dist.numpy(), layer_sens.numpy())[0, 1]
    ax.set_xlabel('edge distance (px) -- 0 == self-loop')
    ax.set_ylabel(f'layer {layer_idx} sensitivity (grad x message)')
    ax.set_title(f'layer {layer_idx}: distance (r = {corr_d:.3f})')

    for j, col in enumerate(sample_cols):
        ax = axes[layer_idx, j + 1]
        values = data['x'][src, x_cols.index(col)].numpy()
        ax.scatter(values, layer_sens.numpy(), s=2, alpha=0.15, linewidths=0)
        corr_c = np.corrcoef(values, layer_sens.numpy())[0, 1]
        ax.set_xlabel(f'{col} (scaled)')
        ax.set_title(f'layer {layer_idx}: {col} (r = {corr_c:.3f})')

fig.suptitle(f'{ANALYSIS_MARKER}: sensitivity vs. distance and feature value, by layer', y=1.0)
fig.tight_layout()
plt.show()

In [ ]:
rng = np.random.default_rng(0)
example_cell_pos = rng.choice(SPLITS[ANALYSIS_SPLIT][2])  # a held-out test cell
mask_edges = (dst == example_cell_pos)
field_pos = src[mask_edges].numpy()  # includes the self-loop -- one of these IS example_cell_pos itself

n_layers = len(layer_sensitivities)
n_panels = n_layers + len(sample_cols)
fig, axes = plt.subplots(1, n_panels, figsize=(6 * n_panels, 6))

for layer_idx, layer_sens in enumerate(layer_sensitivities):
    field_sens = layer_sens[mask_edges].numpy()
    ax = axes[layer_idx]
    sc = ax.scatter(centroids[field_pos, 1], centroids[field_pos, 0], c=field_sens, cmap='viridis', s=100)
    ax.scatter(centroids[example_cell_pos, 1], centroids[example_cell_pos, 0], c='red', s=350, marker='*', label='center cell')
    ax.set_aspect('equal', adjustable='box')
    fig.colorbar(sc, ax=ax, label='sensitivity (grad x message)', shrink=0.4)
    ax.set_title(f'layer {layer_idx}: sensitivity')
    ax.legend()

for j, col in enumerate(sample_cols):
    ax = axes[n_layers + j]
    values = data['x'][field_pos, x_cols.index(col)].numpy()
    sc = ax.scatter(centroids[field_pos, 1], centroids[field_pos, 0], c=values, cmap='coolwarm', s=60)
    ax.scatter(centroids[example_cell_pos, 1], centroids[example_cell_pos, 0], c='red', s=350, marker='*')
    ax.set_aspect('equal', adjustable='box')
    fig.colorbar(sc, ax=ax, label=f'{col} (scaled)', shrink=0.4)
    ax.set_title(col)

fig.suptitle(f'{ANALYSIS_MARKER}: sensitivity by layer vs. feature value (self + neighbors), '
            f'cell {df.index[example_cell_pos]}', y=1.02)
fig.tight_layout()
plt.show()

## Cross-predict on other images

`checkpoint_paths` below points at exactly what the training loop above already
saved (`recipe.load_or_fit`, keyed on `ANALYSIS_SPLIT`) -- nothing new to save here.

Loads each saved checkpoint back (exact `x_cols`, the FITTED scalers reused rather
than refit, the graph radius) and scores it against `EVAL_SOURCES` -- by default
`W9_pattern1`, imaged at the same illumination power in the same session as the
primary well, the only pair in `data/stitched` where a transfer score measures
biology rather than batch (see `tools.dataset.MATCHED_PAIR`).

R2/AUROC on a new well assume `CUSTOM_BACKGROUNDS` holds there too, which is
unverified for anything but the primary well -- Spearman ρ against raw intensity is
the metric that stays meaningful regardless of what threshold a new well would
actually need.

In [ ]:
checkpoint_paths = {marker: f'{RESULTS}/models/{TRAIN_SLUG}_mask_{marker}_{ANALYSIS_SPLIT}{CACHE_SUFFIX}_gnn.pt'
                   for marker in TARGET_MARKERS}

cross_rows, cross_preds = [], {}
for eval_source in EVAL_SOURCES:
    eval_df = load_cell_table(eval_source)
    eval_centroids = eval_df[['centroid_y', 'centroid_x']].to_numpy(np.float32)

    for marker in TARGET_MARKERS:
        model, checkpoint = load_model(checkpoint_paths[marker], IFPredictor)
        log_w = torch.load(checkpoint_paths[marker].replace('.pt', '_calibration.pt'), weights_only=False)['log_w']

        recipe.mask_features(eval_df, f'{MASK_CHANNEL}_bin', max_sigma=recipe.MAX_SIGMA, geometry=True)
        eval_data = apply_prediction_data(eval_df, checkpoint)

        eval_shape = (int(eval_centroids[:,0].max())+2, int(eval_centroids[:,1].max())+2)
        valid_mask = ~border_mask(eval_centroids, eval_shape, checkpoint['radius'])

        pred = calibrated_predict(model, eval_data, log_w)
        y_col = f'{marker}_mean'
        thresh = CUSTOM_BACKGROUNDS.get(y_col)
        y_true = eval_data['y'].numpy().ravel()
        y_positive = (eval_df[y_col].to_numpy(np.float32) > thresh).astype(np.float32) if thresh is not None else None

        met = ev.evaluate(y_true[valid_mask], pred[valid_mask],
                          y_positive[valid_mask] if y_positive is not None else None)
        cross_rows.append(dict(eval_source=eval_source, target=marker, **met))
        cross_preds[(eval_source, marker)] = dict(df=eval_df, pred=pred, y=y_true, valid=valid_mask)
        print(f"{eval_source:30s} {marker:6s} R2={met['r2']:+.4f}  "
              f"AUROC={met.get('auroc', float('nan')):.3f}  spearman={met['spearman']:+.3f}")

cross_results = pd.DataFrame(cross_rows)
cross_results.round(4)

In [ ]:
train_scores = (results[results.split == ANALYSIS_SPLIT]
                .assign(eval_source=f'{TRAIN_SOURCE}  (train, held-out {ANALYSIS_SPLIT} test)'))
comparison = pd.concat([
    train_scores[['eval_source', 'target', 'r2', 'auroc', 'spearman']],
    cross_results[['eval_source', 'target', 'r2', 'auroc', 'spearman']],
], ignore_index=True)
comparison.round(4)

In [ ]:
if EVAL_SOURCES:
    eval_source, marker = EVAL_SOURCES[0], TARGET_MARKERS[0]
    cp = cross_preds[(eval_source, marker)]

    fig = pl.plot_prediction_panel(cp['df'], f'{MASK_CHANNEL}_bin', cp['y'], cp['pred'], target_name=marker,
                                   mask_name=f'{MASK_CHANNEL} mask (input)', smooth_sigma=60)
    fig.suptitle(f'cross-predict: trained on {TRAIN_SOURCE}, evaluated on {eval_source}')
    plt.show()

    spatial = cp['df'][['centroid_x', 'centroid_y']].copy()
    spatial[f'{marker} (actual, scaled)'] = cp['y']
    spatial[f'{marker} (predicted)'] = cp['pred']
    corr_cols = [f'{marker} (actual, scaled)', f'{marker} (predicted)']
    print('spearman:'); print(spatial.loc[cp['valid'], corr_cols].corr(method='spearman').round(3))
    print('\npearson:'); print(spatial.loc[cp['valid'], corr_cols].corr(method='pearson').round(3))

## Read-out

This notebook is meant to be re-run, not just read: swap `TRAIN_SOURCE`/
`EVAL_SOURCES` at the top for any well or any `*_features.parquet` path and every
section below -- training, the feature/correlation/sensitivity views, and the
cross-predict scores -- follows without further edits. For the reasoning behind
*why* this is the recommended config (context scale, the calibration fix, the
density confound), see notebooks 01-04.

In [ ]:
results.to_csv(f'{RESULTS}/05_train_results.csv', index=False)
cross_results.to_csv(f'{RESULTS}/05_cross_predict_results.csv', index=False)
print(f'saved -> {RESULTS}/05_train_results.csv')
print(f'saved -> {RESULTS}/05_cross_predict_results.csv')